In [1]:
import os
import time
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, DataCollatorForTokenClassification,
    DataCollatorWithFlattening
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from sklearn.metrics import (accuracy_score,classification_report, 
confusion_matrix, balanced_accuracy_score, f1_score)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')



device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Modelo de GPU: {torch.cuda.get_device_name(0)}")

semilla = 61298
np.random.seed(semilla)

# Modo offline completo para solucionar problemas al tratar de cargar la configuracion LORA
#os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
#os.environ["HF_HUB_OFFLINE"] = "1"  

GPU disponible: True
Modelo de GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
#Extracción de rutas para la lectura y derivación de archivos
PROJECT_ROOT = Path(os.getcwd()).parent
print(PROJECT_ROOT)
exit()
# Ruta completa a carpetas
MODELOS_PATH = PROJECT_ROOT / 'modelos'
DATOS_PATH = PROJECT_ROOT / 'data' / 'limpieza_final' / 'etiquetado_humano_unificado.csv'
MODELO_TUNEADO_PATH = PROJECT_ROOT / 'modelos'/'Modelo_fold_3'
print(str(MODELOS_PATH))
print(str(MODELO_TUNEADO_PATH))
print(str(DATOS_PATH))
corpus = pd.read_csv(DATOS_PATH, usecols = ['comentario', 'etiquetado_humano','categoria'],encoding = 'utf-8-sig')
print(corpus.head())

C:\Users\nicte\Documents\LLM_PROJECT_1
C:\Users\nicte\Documents\LLM_PROJECT_1\modelos
C:\Users\nicte\Documents\LLM_PROJECT_1\modelos\Modelo_fold_3
C:\Users\nicte\Documents\LLM_PROJECT_1\data\limpieza_final\etiquetado_humano_unificado.csv
                                          comentario  etiquetado_humano  \
0  cuál es el más cercado para rayar el nombre de...                3.0   
1  esos baños deberian estar en el metro no saben...                2.0   
2                           no pues bueno me da risa                3.0   
3                                los van a abandonar                2.0   
4                           nada los tiene contentos                4.0   

         categoria  
0  infraestructura  
1  infraestructura  
2  infraestructura  
3  infraestructura  
4  infraestructura  


In [3]:
modelo = 'nlptown/bert-base-multilingual-uncased-sentiment'
# Primero cargamos el modelo base (sin fine-tuning)
clasificador = AutoModelForSequenceClassification.from_pretrained(
    modelo,
    num_labels = 5,
    id2label= {0:'Negativo',1:'Parcialmente Negativo', 2:'Neutral',3:'Parcialmente Positivo', 4:'Postivo'},
    label2id={'Negativo':0,'Parcialmente Negativo':1, 'Neutral': 2, 'Parcialmente Positivo': 3, 'Positivo':4}
)

# Luego cargamos los adaptadores LoRA (el resultado del tuneo)


clasificador_tuneado = PeftModel.from_pretrained(clasificador,
                                                 str(MODELO_TUNEADO_PATH),
                                                 local_files_only = True,
                                                 config=None,
                                                 cache_dir=None,
                                                 ignore_mismatched_sizes=True,
                                                 force_download=False,
                                                 use_auth_token=None)

# Cargamos el tokenizador de nuestro modelo tuneado
tokenizador = AutoTokenizer.from_pretrained(str(MODELO_TUNEADO_PATH))

# Movemos nuestro modelo a GPU
clasificador_tuneado = clasificador_tuneado.to(device)

print("Clasificador tuneado listo")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Clasificador tuneado listo


In [4]:
comentarios = corpus["comentario"].tolist()

comentarios_tok = tokenizador(comentarios, return_tensors = 'pt', truncation = True, max_lenght = 512, padding = True)
comentarios_tok = {k: v.to(device) for k, v in comentarios_tok.items()}

clasificador_tuneado.eval()

with torch.no_grad():
    outputs = clasificador_tuneado(**comentarios_tok)
    probabilidades = torch.softmax(outputs.logits, dim=-1)
    predicciones = torch.argmax(outputs.logits, dim=-1)
    
    etiquetas = ['Negativo', 'Parcialmente Negativo', 'Neutral', 
                 'Parcialmente Positivo', 'Positivo']
    
resultados = []
for i, texto in enumerate(comentarios):
    resultados.append({
        'comentario': texto,
        'etiqueta estimada': predicciones[i].item(),
        'confianza': torch.max(probabilidades[i]).item()
    })

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


OutOfMemoryError: CUDA out of memory. Tried to allocate 6.14 GiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 10.04 GiB is allocated by PyTorch, and 1.40 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
predicciones = [res['etiqueta estimada'] + 1 for res in resultados]
confianza = [res['confianza'] for res in resultados]

res_df = pd.DataFrame({
    "comentario": muestra["comentario"],
    "etiqueta_estimada": predicciones,
    "etiqueta_real": muestra["etiquetado_humano"],
    "confianza": confianza
})

print(res_df)